In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: torch.rot90(x, 1, [1, 2]))
])

train_data = datasets.EMNIST(root="data", split="digits", train=True, download=True, transform=transform)
test_data  = datasets.EMNIST(root="data", split="digits", train=False, download=True, transform=transform)

train_batch = DataLoader(
    train_data,
    batch_size=256,
    shuffle=True

)

test_batch = DataLoader(
    test_data,
    batch_size=256,
    shuffle=False
)

class MiCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Conv2d(1,32, kernel_size=3)
        self.pool1 =nn.MaxPool2d(2)
        self.layer2 =nn.Conv2d(32,64, kernel_size=3)
        self.pool2 =nn.MaxPool2d(2)
        self.flatten =nn.Flatten()
        self.layerlinear1 =nn.Linear(1600,128)
        self.layerlinear2 =nn.Linear(128,10)
        self.relu =nn.ReLU()

    def forward(self, x):
       x = self.relu(self.layer1(x))
       x = self.pool1(x)
       x = self.relu(self.layer2(x))
       x = self.pool2(x)
       x = self.flatten(x)
       x = self.relu(self.layerlinear1(x))
       x = self.layerlinear2(x)
       return x

model = MiCNN()
model.load_state_dict(torch.load("mnist_model.pth"))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model.train()

for epoch in range(15):
    total_loss = 0
    for images,labels in train_batch:

        predictions = model(images)
        
        loss = criterion(predictions,labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
    
    average_loss = total_loss / len(train_batch)
    print(f"Epoch {epoch + 1}. Loss: {average_loss}")

correct = 0
total   = 0

model.eval()
with torch.no_grad():
    for images, labels in test_batch:
        predictions = model(images)
        predicted   = torch.argmax(predictions, dim=1)
        total      += labels.size(0)
        correct    += (predicted == labels).sum()

accuracy = 100 * correct / total
print(f"Accuracy: {accuracy:.2f}%")

torch.save(model.state_dict(), "mnist_model.pth")
print("Updated model saved!")


        

        

Epoch 1. Loss: 0.11694946940313143
Epoch 2. Loss: 0.032475437941168674
Epoch 3. Loss: 0.024142158938932227
Epoch 4. Loss: 0.01944958137820906
Epoch 5. Loss: 0.015924341560501272
Epoch 6. Loss: 0.013318491812948304
Epoch 7. Loss: 0.011343663608349278
Epoch 8. Loss: 0.009238595933061285
Epoch 9. Loss: 0.00799777590834634
Epoch 10. Loss: 0.006514651233587339
Epoch 11. Loss: 0.005287355666764878
Epoch 12. Loss: 0.00465611323724123
Epoch 13. Loss: 0.004229867254609712
Epoch 14. Loss: 0.003582935420838581
Epoch 15. Loss: 0.0027498916211680636
Accuracy: 99.50%
Updated model saved!


In [8]:
#Functional model
import gradio as gr
import numpy as np
from PIL import Image
import torch
import torch.nn as nn

# Redefine the network class
class MiCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Conv2d(1,32, kernel_size=3)
        self.pool1 =nn.MaxPool2d(2)
        self.layer2 =nn.Conv2d(32,64, kernel_size=3)
        self.pool2 =nn.MaxPool2d(2)
        self.flatten =nn.Flatten()
        self.layerlinear1 =nn.Linear(1600,128)
        self.layerlinear2 =nn.Linear(128,10)
        self.relu =nn.ReLU()

    def forward(self, x):
       x = self.relu(self.layer1(x))
       x = self.pool1(x)
       x = self.relu(self.layer2(x))
       x = self.pool2(x)
       x = self.flatten(x)
       x = self.relu(self.layerlinear1(x))
       x = self.layerlinear2(x)
       return x

# Create model and load saved weights
model = MiCNN()
model.load_state_dict(torch.load("mnist_model.pth"))
model.eval()
print("model ready")

def predict(image):
    if image is None:
        return "Draw a digit"
    
    # get the composite layer which has the drawing
    img = image['composite']
    
    # convert to PIL then grayscale
    img = Image.fromarray(img)
    img = img.convert("L")
    img = img.resize((28, 28), Image.LANCZOS)
    img_array = np.array(img) / 255.0
    
    # invert — MNIST is white on black
    img_array = 1 - img_array
    
    # predict
    tensor = torch.tensor(img_array, dtype=torch.float32).reshape(1, 1, 28, 28)
    
    with torch.no_grad():
        output = model(tensor)
        probs  = torch.softmax(output, dim=1)
        pred   = torch.argmax(probs).item()
    
    return {str(i): float(probs[0][i]) for i in range(10)}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Sketchpad(),
    outputs=gr.Label(num_top_classes=3),
    title="MNIST Digit Recognizer",
    description="Draw a digit and the network will predict it"
)

demo.launch()

model ready
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
